# 03 - Exploratory Data Analysis

**Purpose:** Characterize each cleaned series: seasonality, stationarity, missingness, and distribution shift, flagging leakage-risk implications.

**Inputs:** data/interim/* (from notebook 02)

**Outputs:** EDA figures under results/figures/, notes on leakage-risk implications per dataset.

**Runtime estimate:** 10-20 minutes.

## Time series plots

In [1]:
import pandas as pd

from sdg_leakage_audit.utils.io import read_yaml
from sdg_leakage_audit.utils.paths import configs_dir, data_interim_dir, results_figures_dir
from sdg_leakage_audit.viz.style import apply_plot_style

apply_plot_style()

datasets_cfg = read_yaml(configs_dir() / "datasets.yaml")["datasets"]
figures_dir = results_figures_dir()

interim = {}  # dataset_id -> cleaned DataFrame
target_series = {}  # dataset_id -> representative target Series (single station for Beijing)

for dataset_id, cfg in datasets_cfg.items():
    df = pd.read_parquet(data_interim_dir() / f"{dataset_id}.parquet")
    interim[dataset_id] = df

    if dataset_id == "beijing_multisite_air_quality":
        representative_station = sorted(df["station"].unique())[0]
        series = df.loc[df["station"] == representative_station, cfg["target_column"]]
        print(f"{dataset_id}: using representative station '{representative_station}' for single-series EDA")
    else:
        series = df[cfg["target_column"]]
    target_series[dataset_id] = series.rename(cfg["target_column"])

print(f"Loaded {len(interim)} cleaned datasets.")

from sdg_leakage_audit.viz.plots import plot_time_series

for dataset_id, series in target_series.items():
    plot_time_series(
        series,
        title=f"{dataset_id}: {series.name}",
        save_path=figures_dir / f"03_timeseries_{dataset_id}.png",
    )
print("Saved time series plots for all datasets to results/figures/.")


beijing_multisite_air_quality: using representative station 'Aotizhongxin' for single-series EDA
Loaded 5 cleaned datasets.


2026-10-19 23:40:21 | INFO     | sdg_leakage_audit.viz.plots | Saved figure -> results\figures\03_timeseries_household_power_consumption.png


2026-10-19 23:40:22 | INFO     | sdg_leakage_audit.viz.plots | Saved figure -> results\figures\03_timeseries_appliances_energy_prediction.png


2026-10-19 23:40:22 | INFO     | sdg_leakage_audit.viz.plots | Saved figure -> results\figures\03_timeseries_air_quality_uci.png


2026-10-19 23:40:22 | INFO     | sdg_leakage_audit.viz.plots | Saved figure -> results\figures\03_timeseries_beijing_multisite_air_quality.png


2026-10-19 23:40:23 | INFO     | sdg_leakage_audit.viz.plots | Saved figure -> results\figures\03_timeseries_jena_climate.png


Saved time series plots for all datasets to results/figures/.


## Seasonality and ACF/PACF analysis

In [2]:
from sdg_leakage_audit.viz.plots import plot_acf_pacf

ACF_LAGS = 48  # ~2 days at hourly resolution, ~2 cycles at 10-min resolution's daily period / 5
ACF_MAX_POINTS = 50_000  # ACF/PACF estimation needs no more than this; the 2M-row household
                          # power series times out statsmodels' PACF solver well beyond it

for dataset_id, series in target_series.items():
    acf_series = series.iloc[-ACF_MAX_POINTS:] if len(series) > ACF_MAX_POINTS else series
    plot_acf_pacf(acf_series, lags=ACF_LAGS, save_path=figures_dir / f"03_acf_pacf_{dataset_id}.png")
print("Saved ACF/PACF plots for all datasets to results/figures/.")

print(
    "\nLeakage-risk note: strong autocorrelation at daily/weekly lags (expected for all 5 "
    "series here) means a single observation's near neighbors carry most of its predictive "
    "signal -- this is exactly why P3 (overlapping windows across the train/test boundary) "
    "and P1 (random row/window shuffling) are able to inflate apparent performance: a "
    "near-identical neighbor of a test point can leak into train."
)


2026-10-19 23:40:24 | INFO     | sdg_leakage_audit.viz.plots | Saved figure -> results\figures\03_acf_pacf_household_power_consumption.png


2026-10-19 23:40:25 | INFO     | sdg_leakage_audit.viz.plots | Saved figure -> results\figures\03_acf_pacf_appliances_energy_prediction.png


2026-10-19 23:40:25 | INFO     | sdg_leakage_audit.viz.plots | Saved figure -> results\figures\03_acf_pacf_air_quality_uci.png


2026-10-19 23:40:26 | INFO     | sdg_leakage_audit.viz.plots | Saved figure -> results\figures\03_acf_pacf_beijing_multisite_air_quality.png


2026-10-19 23:40:27 | INFO     | sdg_leakage_audit.viz.plots | Saved figure -> results\figures\03_acf_pacf_jena_climate.png


Saved ACF/PACF plots for all datasets to results/figures/.

Leakage-risk note: strong autocorrelation at daily/weekly lags (expected for all 5 series here) means a single observation's near neighbors carry most of its predictive signal -- this is exactly why P3 (overlapping windows across the train/test boundary) and P1 (random row/window shuffling) are able to inflate apparent performance: a near-identical neighbor of a test point can leak into train.


## ADF/KPSS stationarity tests

In [3]:
from sdg_leakage_audit.stats.tests import stationarity_tests

STATIONARITY_MAX_POINTS = 100_000  # ADF/KPSS lag-order search scales with series length;
                                     # this is far more than enough points for both tests' power

stationarity_rows = []
for dataset_id, series in target_series.items():
    test_series = series.iloc[-STATIONARITY_MAX_POINTS:] if len(series) > STATIONARITY_MAX_POINTS else series
    result = stationarity_tests(test_series, regression="c")
    result["dataset_id"] = dataset_id
    stationarity_rows.append(result)

stationarity_df = pd.DataFrame(stationarity_rows).set_index("dataset_id")
stationarity_df[["adf_pvalue", "kpss_pvalue", "conclusion"]]


,adf_pvalue,kpss_pvalue,conclusion
dataset_id,,,
household_power_consumption,2.138294e-30,0.10,stationary
appliances_energy_prediction,0.000000e+00,0.10,stationary
air_quality_uci,2.497592e-16,0.01,trend_stationary
beijing_multisite_air_quality,0.000000e+00,0.10,stationary
jena_climate,3.719205e-13,0.01,trend_stationary


## Missingness maps

In [4]:
from sdg_leakage_audit.viz.plots import plot_missingness_heatmap

for dataset_id, df in interim.items():
    plot_df = df if dataset_id != "beijing_multisite_air_quality" else df.drop(columns=["station"])
    plot_missingness_heatmap(plot_df, save_path=figures_dir / f"03_missingness_{dataset_id}.png")
print("Saved missingness maps for all datasets to results/figures/.")


2026-10-19 23:41:29 | INFO     | sdg_leakage_audit.viz.plots | Saved figure -> results\figures\03_missingness_household_power_consumption.png


2026-10-19 23:41:30 | INFO     | sdg_leakage_audit.viz.plots | Saved figure -> results\figures\03_missingness_appliances_energy_prediction.png


2026-10-19 23:41:30 | INFO     | sdg_leakage_audit.viz.plots | Saved figure -> results\figures\03_missingness_air_quality_uci.png


2026-10-19 23:41:30 | INFO     | sdg_leakage_audit.viz.plots | Saved figure -> results\figures\03_missingness_beijing_multisite_air_quality.png


2026-10-19 23:41:31 | INFO     | sdg_leakage_audit.viz.plots | Saved figure -> results\figures\03_missingness_jena_climate.png


Saved missingness maps for all datasets to results/figures/.


## Distribution shift: early vs late period

In [5]:
from sdg_leakage_audit.viz.plots import plot_distribution_shift

distribution_shift_rows = []
for dataset_id, series in target_series.items():
    midpoint = series.index.min() + (series.index.max() - series.index.min()) / 2
    early = series[series.index < midpoint]
    late = series[series.index >= midpoint]

    plot_distribution_shift(
        early,
        late,
        title=f"{dataset_id}: early vs late distribution of {series.name}",
        save_path=figures_dir / f"03_distribution_shift_{dataset_id}.png",
    )
    distribution_shift_rows.append(
        {
            "dataset_id": dataset_id,
            "early_mean": early.mean(),
            "late_mean": late.mean(),
            "early_std": early.std(),
            "late_std": late.std(),
            "mean_shift_pct": 100.0 * (late.mean() - early.mean()) / abs(early.mean()) if early.mean() else float("nan"),
        }
    )

distribution_shift_df = pd.DataFrame(distribution_shift_rows).set_index("dataset_id")
distribution_shift_df


2026-10-19 23:41:31 | INFO     | sdg_leakage_audit.viz.plots | Saved figure -> results\figures\03_distribution_shift_household_power_consumption.png


2026-10-19 23:41:32 | INFO     | sdg_leakage_audit.viz.plots | Saved figure -> results\figures\03_distribution_shift_appliances_energy_prediction.png


2026-10-19 23:41:32 | INFO     | sdg_leakage_audit.viz.plots | Saved figure -> results\figures\03_distribution_shift_air_quality_uci.png


2026-10-19 23:41:32 | INFO     | sdg_leakage_audit.viz.plots | Saved figure -> results\figures\03_distribution_shift_beijing_multisite_air_quality.png


2026-10-19 23:41:32 | INFO     | sdg_leakage_audit.viz.plots | Saved figure -> results\figures\03_distribution_shift_jena_climate.png


,early_mean,late_mean,early_std,late_std,mean_shift_pct
dataset_id,,,,,
household_power_consumption,1.104620,1.078380,1.128700,0.979116,-2.375522
appliances_energy_prediction,98.509172,96.880827,108.937504,95.682827,-1.652988
air_quality_uci,1.928690,2.351155,1.219284,1.606953,21.904213
beijing_multisite_air_quality,86.958922,78.621270,79.226278,84.721978,-9.588036
jena_climate,8.813866,10.072593,8.759894,8.017944,14.281201


## Leakage-risk implications

In [6]:
from sdg_leakage_audit.utils.paths import results_tables_dir, to_project_relative

quality_summary = stationarity_df.join(distribution_shift_df)
out_path = results_tables_dir() / "eda_summary.csv"
quality_summary.to_csv(out_path)
print(f"Saved EDA summary table -> {to_project_relative(out_path)}")

print(
    "\nPer-dataset leakage-risk implications (derived from the analyses above):\n"
    "- Any dataset flagged 'non_stationary' or showing a large mean_shift_pct between "
    "early/late periods is exactly where a full-data-fit scaler/imputer (P2) or a "
    "target-derived feature computed over the whole series (P4) will most overstate "
    "performance, because the fitted statistic silently encodes future-period information "
    "that a causal train-only fit could never have seen.\n"
    "- Strong short-lag autocorrelation (see ACF/PACF) is why overlapping windows across "
    "the train/test boundary (P3) and shuffled splits (P1, P6) leak: nearby points are "
    "highly redundant, so a 'test' point's near-duplicate neighbor sitting in 'train' lets "
    "the model partially memorize rather than forecast.\n"
    "- Datasets with non-trivial missingness after frequency enforcement (see notebook 02) "
    "compound this risk further: naive full-series imputation is itself a form of P2 if not "
    "restricted to the train fold."
)


Saved EDA summary table -> results\tables\eda_summary.csv

Per-dataset leakage-risk implications (derived from the analyses above):
- Any dataset flagged 'non_stationary' or showing a large mean_shift_pct between early/late periods is exactly where a full-data-fit scaler/imputer (P2) or a target-derived feature computed over the whole series (P4) will most overstate performance, because the fitted statistic silently encodes future-period information that a causal train-only fit could never have seen.
- Strong short-lag autocorrelation (see ACF/PACF) is why overlapping windows across the train/test boundary (P3) and shuffled splits (P1, P6) leak: nearby points are highly redundant, so a 'test' point's near-duplicate neighbor sitting in 'train' lets the model partially memorize rather than forecast.
- Datasets with non-trivial missingness after frequency enforcement (see notebook 02) compound this risk further: naive full-series imputation is itself a form of P2 if not restricted to the 

## Findings / what to carry forward

- All 5 series show strong short-lag autocorrelation (see `results/figures/03_acf_pacf_*.png`)
  plus daily/weekly seasonal structure -- exactly the setting where overlapping-window (P3)
  and shuffled-split (P1/P6) leakage can inflate apparent performance, since nearby points are
  highly redundant.
- Stationarity conclusions and early-vs-late distribution shift are in
  `results/tables/eda_summary.csv`; datasets with the largest `mean_shift_pct` are the ones
  where a full-data-fit scaler/imputer (P2) or whole-series target-derived feature (P4) is
  expected to inflate performance the most in notebook 07 -- a concrete, testable prediction
  to check inflation results against.
- Missingness maps (`results/figures/03_missingness_*.png`) confirm the gap patterns already
  quantified numerically in notebook 02's `data_quality_report.csv`; no new missingness issues
  surfaced visually.
- Carry forward to notebook 04 (feature engineering): the ACF/PACF lag structure here should
  directly inform which lag/rolling-window lengths are worth engineering as features.
